# Regresión Lineal Múltiple
### Predicción del costo del seguro médico

**Variables de entrada (X):** Edad, BMI, Fumador, Género  
**Variable a predecir (Y):** Costo del seguro (charges)

La regresión lineal múltiple encuentra la relación entre varias variables independientes
y una variable dependiente usando la ecuación:

```
Y = w1*X1 + w2*X2 + w3*X3 + w4*X4 + b
```

In [1]:
# ============================================================
# 1. IMPORTAR LIBRERÍAS
# ============================================================
# pandas  → manejar datos en tablas (DataFrame)
# numpy   → operaciones matemáticas con arreglos
# sklearn → herramientas de Machine Learning

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [2]:
# ============================================================
# 2. CARGAR EL DATASET
# ============================================================
# El archivo insurance.csv contiene datos de 1338 personas
# Columnas: age, sex, bmi, children, smoker, region, charges

df = pd.read_csv("insurance.csv")

print("Primeras filas del dataset:")
print(df.head())

print("\nDimensiones del dataset (filas, columnas):")
print(df.shape)

Primeras filas del dataset:
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520

Dimensiones del dataset (filas, columnas):
(1338, 7)


In [3]:
# ============================================================
# 3. SELECCIONAR Y RENOMBRAR COLUMNAS
# ============================================================
# Solo usamos las columnas que nos pide la tarea:
# age (edad), bmi, smoker (fumador), sex (género), charges (cargos)

df_regresion = df[['age', 'bmi', 'smoker', 'sex', 'charges']].copy()

# Renombramos al español para mayor claridad
df_regresion.columns = ['edad', 'bmi', 'fumador', 'genero', 'cargos']

print("DataFrame con columnas seleccionadas:")
print(df_regresion.head())

DataFrame con columnas seleccionadas:
   edad     bmi fumador  genero       cargos
0    19  27.900     yes  female  16884.92400
1    18  33.770      no    male   1725.55230
2    28  33.000      no    male   4449.46200
3    33  22.705      no    male  21984.47061
4    32  28.880      no    male   3866.85520


In [4]:
# ============================================================
# 4. BINARIZACIÓN (convertir texto → número)
# ============================================================
# Los modelos de ML solo entienden números, no texto.
# Convertimos las columnas categóricas (texto) a binario:
#
#   fumador:  'yes' → 1  |  'no' → 0
#   genero:   'male' → 1 |  'female' → 0
#
# Esto se llama binarización (o Label Encoding para variables de 2 valores).
# Si una columna tuviera 3+ categorías (ej: región: norte/sur/este/oeste),
# usaríamos One-Hot Encoding con pd.get_dummies() en su lugar.

df_regresion['fumador'] = df_regresion['fumador'].map({
    'yes': 1,
    'no': 0
})

df_regresion['genero'] = df_regresion['genero'].map({
    'male': 1,
    'female': 0
})

print("DataFrame después de binarización:")
print(df_regresion.head())
print("\nTipos de datos:")
print(df_regresion.dtypes)

DataFrame después de binarización:
   edad     bmi  fumador  genero       cargos
0    19  27.900        1       0  16884.92400
1    18  33.770        0       1   1725.55230
2    28  33.000        0       1   4449.46200
3    33  22.705        0       1  21984.47061
4    32  28.880        0       1   3866.85520

Tipos de datos:
edad         int64
bmi        float64
fumador      int64
genero       int64
cargos     float64
dtype: object


In [5]:
# ============================================================
# 5. SEPARAR VARIABLES X e Y
# ============================================================
# X = variables independientes (entradas del modelo)
# y = variable dependiente (lo que queremos predecir)

X = df_regresion[['edad', 'bmi', 'fumador', 'genero']].values
y = df_regresion['cargos'].values

print("Forma de X:", X.shape)   # (1338 filas, 4 columnas)
print("Forma de y:", y.shape)   # (1338 valores)

Forma de X: (1338, 4)
Forma de y: (1338,)


In [6]:
# ============================================================
# 6. NORMALIZACIÓN (Min-Max Scaling)
# ============================================================
# Escalar los datos al rango [0, 1] para que ninguna variable
# domine a las demás por tener valores más grandes.
# Fórmula: X_norm = (X - X_min) / (X_max - X_min)
#
# Nota: fumador y genero ya son 0 o 1, no necesitan normalizarse.
# Solo normalizamos edad, bmi y los cargos (y).

# --- Normalizar EDAD ---
edad_max = np.amax(X[:, 0])
edad_min = np.amin(X[:, 0])
X[:, 0] = (X[:, 0] - edad_min) / (edad_max - edad_min)

# --- Normalizar BMI ---
bmi_max = np.amax(X[:, 1])
bmi_min = np.amin(X[:, 1])
X[:, 1] = (X[:, 1] - bmi_min) / (bmi_max - bmi_min)

# --- Normalizar CARGOS (y) ---
y_max = np.amax(y)
y_min = np.amin(y)
y = (y - y_min) / (y_max - y_min)

# Mostrar datos normalizados
df_normalizado = pd.DataFrame({
    'edad_norm':    X[:, 0],
    'bmi_norm':     X[:, 1],
    'fumador':      X[:, 2],
    'genero':       X[:, 3],
    'cargos_norm':  y
})

print("Datos normalizados:")
print(df_normalizado.head())

Datos normalizados:
   edad_norm  bmi_norm  fumador  genero  cargos_norm
0   0.021739  0.321227      1.0     0.0     0.251611
1   0.000000  0.479150      0.0     1.0     0.009636
2   0.217391  0.458434      0.0     1.0     0.053115
3   0.326087  0.181464      0.0     1.0     0.333010
4   0.304348  0.347592      0.0     1.0     0.043816


In [7]:
# ============================================================
# 7. DIVIDIR EN ENTRENAMIENTO Y PRUEBA
# ============================================================
# El 80% de los datos se usa para entrenar el modelo.
# El 20% restante se reserva para evaluar qué tan bien predice
# con datos que nunca ha visto.
#
# random_state=42 → fija el "azar" para que siempre se dividan igual.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Tamaño entrenamiento:", X_train.shape[0], "filas")
print("Tamaño prueba:       ", X_test.shape[0],  "filas")

Tamaño entrenamiento: 1070 filas
Tamaño prueba:        268 filas


In [8]:
# ============================================================
# 8. CREAR Y ENTRENAR EL MODELO
# ============================================================
# LinearRegression de sklearn encuentra los mejores coeficientes (w)
# usando el método de Mínimos Cuadrados.

modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Ver los coeficientes aprendidos
print("Intercepto (b):")
print(round(modelo.intercept_, 4))

print("\nCoeficientes (w) aprendidos:")
print("  Edad:    ", round(modelo.coef_[0], 4))
print("  BMI:     ", round(modelo.coef_[1], 4))
print("  Fumador: ", round(modelo.coef_[2], 4))
print("  Género:  ", round(modelo.coef_[3], 4))

print("\n→ El valor mayor en coeficiente = mayor impacto en el precio.")

Intercepto (b):
-0.0472

Coeficientes (w) aprendidos:
  Edad:     0.1905
  BMI:      0.1937
  Fumador:  0.3779
  Género:   0.0001

→ El valor mayor en coeficiente = mayor impacto en el precio.


In [9]:
# ============================================================
# 9. EVALUAR EL MODELO (Métricas)
# ============================================================
# Predecimos sobre el conjunto de prueba y comparamos contra los valores reales.
#
# MAE  → Error promedio en la misma unidad que los datos (más fácil de interpretar)
# MSE  → Penaliza más los errores grandes (sensible a outliers)
# R²   → Qué tanto explica el modelo la variación de los datos
#         0 = no explica nada | 1 = explica todo perfectamente

predicciones = modelo.predict(X_test)

mae = mean_absolute_error(y_test, predicciones)
mse = mean_squared_error(y_test, predicciones)
r2  = r2_score(y_test, predicciones)

print("============================")
print("MÉTRICAS DEL MODELO")
print("============================")
print("MAE:", round(mae, 4))
print("MSE:", round(mse, 4))
print("R²: ", round(r2,  4))
print("\nInterpretación R²:")
print(f"  El modelo explica el {round(r2*100, 1)}% de la variación en el costo del seguro.")

MÉTRICAS DEL MODELO
MAE: 0.068
MSE: 0.0088
R²:  0.7777

Interpretación R²:
  El modelo explica el 77.8% de la variación en el costo del seguro.


In [10]:
# ============================================================
# 10. PREDECIR PARA UN NUEVO CLIENTE
# ============================================================
# Ejemplo: persona de 30 años, BMI 28, fumadora, masculino

edad_nueva   = 30
bmi_nuevo    = 28
fumador_nuevo = 1   # 1 = sí fuma
genero_nuevo  = 1   # 1 = masculino | 0 = femenino

# Normalizamos con los mismos min/max del dataset original
edad_norm = (edad_nueva - edad_min) / (edad_max - edad_min)
bmi_norm  = (bmi_nuevo  - bmi_min)  / (bmi_max  - bmi_min)

nuevo_cliente = [[edad_norm, bmi_norm, fumador_nuevo, genero_nuevo]]

# Predecir (resultado normalizado)
prediccion_norm = modelo.predict(nuevo_cliente)

# Desnormalizar → regresar a dólares reales
prediccion_real = (prediccion_norm[0] * (y_max - y_min)) + y_min

print("============================")
print("NUEVO CLIENTE")
print("============================")
print("Edad:   ", edad_nueva)
print("BMI:    ", bmi_nuevo)
print("Fumador:", "Sí" if fumador_nuevo == 1 else "No")
print("Género: ", "Masculino" if genero_nuevo == 1 else "Femenino")
print("\nCosto estimado del seguro: $", round(prediccion_real, 2))

NUEVO CLIENTE
Edad:    30
BMI:     28
Fumador: Sí
Género:  Masculino

Costo estimado del seguro: $ 28894.41
